# 🌍 Deprem Haritası Analizi

## Proje Amacı

"""Bu projede USGS (United States Geological Survey) tarafından sunulan ücretsiz deprem verileri kullanılarak son 30 günde meydana gelen depremler analiz edilmiştir.

Proje kapsamında:

- USGS API'den canlı veri çekilmiştir.
- Veriler pandas ile işlenmiştir.
- Folium kullanılarak etkileşimli deprem haritası oluşturulmuştur.
- Günlük deprem sayıları grafik ile görselleştirilmiştir.
- Harita HTML formatında dışa aktarılmıştır.

---"""

# 1. Gerekli Kütüphanelerin Yüklenmesi

Bu bölümde projede kullanılacak Python kütüphaneleri içe aktarılmaktadır.

In [ ]:
import requests

# 2. USGS API'den Deprem Verilerinin Çekilmesi

USGS tarafından ücretsiz olarak sunulan GeoJSON API kullanılarak son 30 güne ait deprem verileri internet üzerinden alınmaktadır.

In [ ]:
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"

response = requests.get(url)

print(response.status_code)

# API Yanıtının JSON Formatına Dönüştürülmesi

API'den gelen yanıt JSON formatına dönüştürülerek Python tarafından işlenebilir hale getirilmektedir.

In [ ]:
data = response.json()

type(data)

JSON Yapısının İncelenmesi
Veri yapısında bulunan ana bölümler incelenmektedir.

In [ ]:
print(data.keys())

## İlk Deprem Kaydının İncelenmesi

API'den gelen ilk deprem kaydı görüntülenerek kullanılacak alanlar belirlenmektedir.

In [ ]:
print(data["features"][0])

In [ ]:
ilk_deprem = data["features"][0]

print("Büyüklük:", ilk_deprem["properties"]["mag"])
print("Yer:", ilk_deprem["properties"]["place"])
print("Tarih:", ilk_deprem["properties"]["time"])
print("Enlem:", ilk_deprem["geometry"]["coordinates"][1])
print("Boylam:", ilk_deprem["geometry"]["coordinates"][0])
print("Derinlik:", ilk_deprem["geometry"]["coordinates"][2])

# 3. Verilerin DataFrame'e Hazırlanması

Bu bölümde API'den gelen tüm deprem kayıtları dolaşılarak tablo oluşturulmaya uygun hale getirilmektedir.

In [ ]:
depremler = []

In [ ]:
for deprem in data["features"]:

    bilgiler = deprem["properties"]
    koordinatlar = deprem["geometry"]["coordinates"]

    depremler.append({
        "Tarih": bilgiler["time"],
        "Büyüklük": bilgiler["mag"],
        "Yer": bilgiler["place"],
        "Enlem": koordinatlar[1],
        "Boylam": koordinatlar[0],
        "Derinlik": koordinatlar[2]
    })

In [ ]:
len(depremler)

## Pandas DataFrame Oluşturulması

Elde edilen deprem kayıtları pandas DataFrame yapısına dönüştürülmektedir.

In [ ]:
import pandas as pd

df = pd.DataFrame(depremler)

# 4. Veri Ön İşleme

Tarih sütunu okunabilir tarih-saat formatına dönüştürülmektedir.

In [ ]:
df["Tarih"] = pd.to_datetime(df["Tarih"], unit="ms")

# 5. Etkileşimli Haritanın Oluşturulması

Folium kütüphanesi kullanılarak deprem verilerinin dünya haritası üzerinde görselleştirilmesi için gerekli ortam hazırlanmaktadır.

In [ ]:
!pip install folium

In [ ]:
import folium

## Temel Haritanın Oluşturulması

İlk olarak dünya haritası oluşturulmaktadır.

In [ ]:
harita = folium.Map(
    location=[20, 0],
    zoom_start=2
)

harita

## İlk Deprem Noktasının Haritaya Eklenmesi

Haritanın doğru çalıştığını test etmek amacıyla ilk deprem kaydı harita üzerinde gösterilmektedir.

In [ ]:
ilk = df.iloc[0]

folium.CircleMarker(
    location=[ilk["Enlem"], ilk["Boylam"]],
    radius=5,
    popup=f"""
    <b>Yer:</b> {ilk["Yer"]}<br>
    <b>Büyüklük:</b> {ilk["Büyüklük"]}<br>
    <b>Tarih:</b> {ilk["Tarih"]}
    """,
    color="red",
    fill=True,
    fill_color="red"
).add_to(harita)

harita

## Tüm Depremlerin Haritaya Eklenmesi

DataFrame içerisinde bulunan tüm deprem kayıtları dolaşılarak büyüklüklerine göre renkli ve etkileşimli daireler harita üzerine eklenmektedir.

In [ ]:
# Yeni bir harita oluşturuyoruz
harita = folium.Map(location=[20, 0], zoom_start=2)

# Tüm depremleri dolaşıyoruz
for _, satir in df.iterrows():

    # Büyüklük bilgisi boşsa atla
    if pd.isna(satir["Büyüklük"]):
        continue

    # Dairenin büyüklüğünü deprem büyüklüğüne göre ayarla
    yaricap = max(satir["Büyüklük"] * 2, 2)

    folium.CircleMarker(
        location=[satir["Enlem"], satir["Boylam"]],
        radius=yaricap,
        popup=f"""
        <b>Yer:</b> {satir['Yer']}<br>
        <b>Büyüklük:</b> {satir['Büyüklük']}<br>
        <b>Tarih:</b> {satir['Tarih']}
        """,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.6
    ).add_to(harita)

# Haritayı göster
harita

# Haritada renk değişikliğine gidildi.

In [ ]:
import folium

# Dünya haritası
harita = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="CartoDB positron"
)

# Renk belirleme fonksiyonu
def deprem_rengi(mag):

    if mag < 2:
        return "#4CAF50"      # Yeşil

    elif mag < 4:
        return "#FFC107"      # Sarı

    elif mag < 5.5:
        return "#FF9800"      # Turuncu

    elif mag < 7:
        return "#F44336"      # Kırmızı

    else:
        return "#9C27B0"      # Mor


for _, satir in df.iterrows():

    if pd.isna(satir["Büyüklük"]):
        continue

    folium.CircleMarker(

        location=[satir["Enlem"], satir["Boylam"]],

        radius=max(satir["Büyüklük"]*1.8,2),

        color=deprem_rengi(satir["Büyüklük"]),

        fill=True,

        fill_color=deprem_rengi(satir["Büyüklük"]),

        fill_opacity=0.45,

        weight=1,

        popup=f"""
        <b>📍 Yer:</b> {satir['Yer']}<br>
        <b>📈 Büyüklük:</b> {satir['Büyüklük']}<br>
        <b>📅 Tarih:</b> {satir['Tarih']}<br>
        <b>📏 Derinlik:</b> {satir['Derinlik']} km
        """

    ).add_to(harita)

harita

# 6. Haritanın HTML Olarak Kaydedilmesi

Oluşturulan etkileşimli harita HTML dosyası olarak kaydedilmektedir.

In [ ]:
harita.save("deprem_haritasi.html")

In [ ]:
from google.colab import files

files.download("deprem_haritasi.html")

# 7. Günlük Deprem Analizi

Deprem kayıtları gün bazında gruplanarak günlük deprem sayıları hesaplanmaktadır.

In [ ]:
# Tarihten sadece gün bilgisini al
df["Gün"] = df["Tarih"].dt.date

# Her gün kaç deprem olmuş?
gunluk = df.groupby("Gün").size()

gunluk.head()

## Günlük Deprem Sayısının Görselleştirilmesi

Günlük deprem sayıları çizgi grafik ile görselleştirilmektedir.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))

plt.plot(
    gunluk.index,
    gunluk.values,
    marker="o",
    linewidth=2
)

plt.title("Son 30 Günlük Deprem Sayısı", fontsize=16)
plt.xlabel("Tarih")
plt.ylabel("Deprem Sayısı")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Haritanın Görsel Olarak Geliştirilmesi

Deprem büyüklüklerine göre farklı renkler atanarak haritanın okunabilirliği artırılmaktadır.

In [ ]:
import folium
from folium.plugins import Fullscreen

harita = folium.Map(
    location=[20,0],
    zoom_start=2,
    tiles=None
)

# Harita temaları
folium.TileLayer(
    "OpenStreetMap",
    name="Normal Harita"
).add_to(harita)

folium.TileLayer(
    "CartoDB positron",
    name="Açık Tema"
).add_to(harita)

folium.TileLayer(
    "CartoDB dark_matter",
    name="Koyu Tema"
).add_to(harita)

folium.TileLayer(
    "Esri.WorldImagery",
    attr="Esri",
    name="Uydu Görünümü"
).add_to(harita)

Fullscreen().add_to(harita)

In [ ]:
def deprem_rengi(mag):

    if mag < 2:
        return "#2ECC71"

    elif mag < 4:
        return "#F1C40F"

    elif mag < 5.5:
        return "#E67E22"

    elif mag < 7:
        return "#E74C3C"

    else:
        return "#8E44AD"

In [ ]:
for _, satir in df.iterrows():

    if pd.isna(satir["Büyüklük"]):
        continue

    folium.CircleMarker(

        location=[satir["Enlem"], satir["Boylam"]],

        radius=max(satir["Büyüklük"]*1.8,2),

        color=deprem_rengi(satir["Büyüklük"]),

        fill=True,

        fill_color=deprem_rengi(satir["Büyüklük"]),

        fill_opacity=0.60,

        weight=1,

        popup=f"""
        <h4>Deprem Bilgisi</h4>

        <b>Yer:</b> {satir["Yer"]}<br>

        <b>Büyüklük:</b> {satir["Büyüklük"]}<br>

        <b>Tarih:</b> {satir["Tarih"]}<br>

        <b>Derinlik:</b> {satir["Derinlik"]} km
        """

    ).add_to(harita)

# 8. Haritanın Görsel Olarak Geliştirilmesi

Deprem büyüklüklerine göre farklı renkler atanarak haritanın okunabilirliği artırılmaktadır.

In [ ]:
folium.LayerControl(collapsed=False).add_to(harita)

# 9. Sonuç

Oluşturulan etkileşimli deprem haritası aşağıda görüntülenmektedir. Harita üzerinde bulunan her bir nokta bir depremi temsil etmekte olup, noktaya tıklandığında depremin yeri, büyüklüğü, tarihi ve derinliği görüntülenebilmektedir.

In [ ]:
harita